# 📓 Notebook 13 — Toolformer Style Agents---**Phase 5 · Advanced Deep Agent Concepts**> Standard agents have fixed tools. Toolformer-style agents DISCOVER and SELECT tools dynamically.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Dynamic tool discovery | Register tools at runtime || Tool selection | LLM picks tools based on capability matching || Conditional execution | Only use tools when beneficial || Tool composition | Chain tools into workflows |### 🏗️ Build: Agent with Dynamic Tool Discovery

## 1. Setup

In [ ]:
import os, json, math, timefrom dotenv import load_dotenvimport litellmload_dotenv()DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "gpt-4o-mini")print(f"🤖 {DEFAULT_MODEL}")

---## 2. Dynamic Tool RegistryInstead of hardcoding tools, we build a registry that can grow at runtime.

In [ ]:
class ToolRegistry:    """Dynamic tool registration and discovery system."""        def __init__(self):        self.tools = {}     # name → {"schema": ..., "function": ..., "metadata": ...}        def register(self, name, description, parameters, function, tags=None):        """Register a new tool at runtime."""        schema = {            "type": "function",            "function": {                "name": name,                "description": description,                "parameters": parameters,            }        }        self.tools[name] = {            "schema": schema,            "function": function,            "tags": tags or [],            "call_count": 0,        }        print(f"  ✅ Registered: {name} (tags: {tags})")        def unregister(self, name):        if name in self.tools:            del self.tools[name]            print(f"  ❌ Unregistered: {name}")        def get_schemas(self, tags=None):        """Get tool schemas, optionally filtered by tags."""        if tags:            return [t["schema"] for t in self.tools.values()                     if any(tag in t["tags"] for tag in tags)]        return [t["schema"] for t in self.tools.values()]        def execute(self, name, **kwargs):        """Execute a registered tool."""        if name not in self.tools:            return json.dumps({"error": f"Tool '{name}' not found"})        self.tools[name]["call_count"] += 1        try:            return self.tools[name]["function"](**kwargs)        except Exception as e:            return json.dumps({"error": str(e)})        def list_tools(self):        for name, t in self.tools.items():            print(f"  🔧 {name} [{', '.join(t['tags'])}] (calls: {t['call_count']})")        def stats(self):        total_calls = sum(t["call_count"] for t in self.tools.values())        print(f"  📊 {len(self.tools)} tools registered, {total_calls} total calls")# Create registry and register tools dynamicallyregistry = ToolRegistry()registry.register(    name="calculate",    description="Perform mathematical calculations",    parameters={"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]},    function=lambda expression: json.dumps({"result": eval(expression, {"__builtins__": {}, "math": math}, {"sqrt": math.sqrt, "pi": math.pi, "sin": math.sin, "cos": math.cos})}),    tags=["math", "core"])registry.register(    name="string_transform",    description="Transform strings: reverse, uppercase, lowercase, title case, word count",    parameters={"type": "object", "properties": {"text": {"type": "string"}, "operation": {"type": "string", "enum": ["reverse", "upper", "lower", "title", "word_count"]}}, "required": ["text", "operation"]},    function=lambda text, operation: json.dumps({"result": {"reverse": text[::-1], "upper": text.upper(), "lower": text.lower(), "title": text.title(), "word_count": len(text.split())}.get(operation, "unknown operation")}),    tags=["text", "core"])registry.register(    name="date_info",    description="Get current date/time information",    parameters={"type": "object", "properties": {}, "required": []},    function=lambda: json.dumps({"date": time.strftime("%Y-%m-%d"), "time": time.strftime("%H:%M:%S"), "day": time.strftime("%A")}),    tags=["time", "core"])registry.list_tools()

In [ ]:
# Register more tools dynamicallyregistry.register(    name="json_validator",    description="Validate if a string is valid JSON and return parsed structure",    parameters={"type": "object", "properties": {"json_string": {"type": "string"}}, "required": ["json_string"]},    function=lambda json_string: json.dumps({"valid": True, "parsed": json.loads(json_string)}) if json_dumps_valid(json_string) else json.dumps({"valid": False}),    tags=["data", "validation"])def json_dumps_valid(s):    try:        json.loads(s)        return True    except:        return Falseregistry.register(    name="list_operations",    description="Perform operations on comma-separated lists: sort, unique, count, reverse, shuffle",    parameters={"type": "object", "properties": {"items": {"type": "string", "description": "Comma-separated values"}, "operation": {"type": "string", "enum": ["sort", "unique", "count", "reverse"]}}, "required": ["items", "operation"]},    function=lambda items, operation: json.dumps({"result": {"sort": sorted(items.split(",")), "unique": list(set(items.split(","))), "count": len(items.split(",")), "reverse": items.split(",")[::-1]}.get(operation.strip(), "unknown")}),    tags=["data", "core"])print(f"\n📊 Registry now has {len(registry.tools)} tools")

In [ ]:
class ToolformerAgent:    """Agent that dynamically discovers and uses tools from a registry."""        def __init__(self, registry, model=None):        self.registry = registry        self.model = model or DEFAULT_MODEL        def _should_use_tools(self, query):        """Determine if tools would be beneficial for this query."""        response = litellm.completion(            model=self.model,            messages=[{                "role": "user",                "content": f"Can this question be answered better with external tools (math, search, data processing) or is it purely a knowledge question? Answer 'TOOLS' or 'KNOWLEDGE'.\n\nQuestion: {query}"            }],            temperature=0, max_tokens=10,        )        return "tool" in response.choices[0].message.content.lower()        def _select_tools(self, query):        """Select relevant tools for the query."""        all_schemas = self.registry.get_schemas()        tool_descriptions = "\n".join([            f"- {s['function']['name']}: {s['function']['description']}"             for s in all_schemas        ])                response = litellm.completion(            model=self.model,            messages=[{                "role": "user",                "content": f"Which tools are relevant for this task? Available:\n{tool_descriptions}\n\nTask: {query}\n\nReturn JSON with key 'tools' containing array of tool names."            }],            response_format={"type": "json_object"},            temperature=0,        )        try:            selected = json.loads(response.choices[0].message.content).get("tools", [])            return [s for s in all_schemas if s["function"]["name"] in selected]        except:            return all_schemas        def run(self, query, verbose=True):        """Run with dynamic tool selection."""        # Step 1: Should we use tools?        use_tools = self._should_use_tools(query)                if verbose:            print(f"  🤔 Use tools? {'Yes' if use_tools else 'No (knowledge-only)'}")                if not use_tools:            response = litellm.completion(                model=self.model,                messages=[{"role": "user", "content": query}],                temperature=0.5, max_tokens=500,            )            return response.choices[0].message.content                # Step 2: Select relevant tools        selected_tools = self._select_tools(query)        if verbose:            print(f"  🔧 Selected: {[t['function']['name'] for t in selected_tools]}")                # Step 3: Run with selected tools        messages = [            {"role": "system", "content": "Use the available tools to help answer the query accurately."},            {"role": "user", "content": query}        ]                for round_num in range(5):            response = litellm.completion(                model=self.model, messages=messages,                tools=selected_tools, tool_choice="auto", temperature=0,            )            message = response.choices[0].message                        if not message.tool_calls:                return message.content                        messages.append(message)            for tc in message.tool_calls:                args = json.loads(tc.function.arguments)                result = self.registry.execute(tc.function.name, **args)                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})                if verbose:                    print(f"  🔧 {tc.function.name} → {result[:60]}...")                return "Max rounds exceeded"# Test the Toolformer agenttf_agent = ToolformerAgent(registry)print("🤖 Toolformer Agent Demo")print("=" * 60)queries = [    "What is the meaning of life?",                      # Knowledge only    "Calculate the area of a circle with radius 7.5",     # Needs math tool    "Reverse the string 'Hello World'",                   # Needs string tool    "Sort these items: banana, apple, cherry, date",      # Needs list tool    "What day is it today?",                              # Needs date tool]for q in queries:    print(f"\n❓ {q}")    answer = tf_agent.run(q)    print(f"📝 {answer[:150]}")    print("-" * 40)registry.stats()

---## 📝 Interview Quiz — Toolformer Agents### Q1: What makes Toolformer different from a standard tool-using agent?<details><summary>💡 Answer</summary>Standard agent: Fixed set of tools defined at design time.Toolformer: Dynamically discovers, selects, and uses tools at runtime.Key differences:- **Discovery**: Can find new tools from a registry/catalog- **Selection**: Decides which tools are relevant per query (doesn't pass all tools)- **Conditional use**: Decides IF tools are needed at all- **Composition**: Can chain tools it hasn't been explicitly trained on</details>### Q2: Why not give the LLM ALL available tools every time?<details><summary>💡 Answer</summary>Problems with passing all tools:1. **Context window** — Each tool schema costs tokens (50-200 per tool)2. **Selection accuracy** — More options = more confusion = wrong tool chosen3. **Cost** — Every token in the prompt costs money4. **Latency** — Larger prompts take longer to processBest practice: Pre-filter tools (by tags, query analysis, or embedding similarity) and only pass 3-5 most relevant tools.</details>### Q3: How would you build a tool marketplace for agents?<details><summary>💡 Answer</summary>1. **Registry**: Central store of tool schemas + implementations2. **Discovery**: Embedding-based search over tool descriptions3. **Versioning**: SemVer for tool interfaces4. **Authentication**: Each tool may require different API keys5. **Rating**: Track success rate per tool6. **Sandboxing**: Run untrusted tools in isolated environments7. **Documentation**: Auto-generated docs from schemasSimilar to: npm for JavaScript, pip for Python, but for agent tools.</details>

---## ✅ Notebook 13 Summary| Concept | Key Takeaway ||---------|-------------|| Dynamic registry | Register/unregister tools at runtime || Tool selection | Pre-filter relevant tools per query || Conditional execution | Only use tools when beneficial || Tool composition | Chain discovered tools for complex tasks || Scalability | Don't pass all tools; pre-filter by relevance |### ➡️ Next: [Notebook 14 — Filesystem Agents](./14_filesystem_agent.ipynb)